In [1]:
!pip install transformers sentencepiece sacrebleu pandas tqdm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Загрузка датасета и готовых сегментов Wisper

In [2]:
from pathlib import Path
from google.colab import drive
import pickle

drive.mount('/content/drive', force_remount=True)

dataset_path = Path("/content/drive/MyDrive/diplom/dataset")

with open("/content/drive/MyDrive/diplom/whisper_results_segments.pkl", "rb") as f:
    whisper_results = pickle.load(f)

print("Загружено видео:", len(whisper_results))
print("Ключи первого элемента:", whisper_results[0].keys())
print("Папки датасета:", [p.name for p in dataset_path.iterdir()])

Mounted at /content/drive
Загружено видео: 15
Ключи первого элемента: dict_keys(['folder', 'video', 'audio', 'ref', 'raw_text', 'segments', 'asr_time_sec'])
Папки датасета: ['video1', 'video2', 'video3', 'video4', 'video5', 'video6', 'video7', 'video8', 'video9', 'video10', 'video11', 'video12', 'video13', 'video14', 'video15']


Базовая сегментация на субтитры

In [3]:
import re
import textwrap
import time
from typing import List

MAX_LINE_LEN = 39
MAX_LINES = 2
MAX_CHARS_BLOCK = MAX_LINE_LEN * MAX_LINES
MAX_READING_SPEED = 17
MIN_BLOCK_DUR = 1.0
MIN_GAP_BETWEEN_SUBS = 1.0



def clean_text_for_subs(text: str) -> str:
    text = re.sub(r"\b(\w+)([\s,]+\1\b)+", r"\1", text, flags=re.IGNORECASE)
    fillers = r"\b(er|eh|hmm|mm-hmm|uh-huh|uh-uh|oh|ah|uh|huh|erm|um)\b[,\s]*"
    text = re.sub(fillers, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text


def split_text_into_lines(text: str, max_line_len: int = MAX_LINE_LEN) -> List[str]:
    return textwrap.wrap(text, width=max_line_len)


def is_valid_block_text(
    text: str,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
) -> bool:
    lines = split_text_into_lines(text, max_line_len=max_line_len)
    return (
        len(text) <= max_chars_block
        and len(lines) <= max_lines
        and all(len(line) <= max_line_len for line in lines)
    )


def split_segment_text_into_blocks(
    text: str,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
) -> List[str]:
    text = clean_text_for_subs(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    blocks = []

    for sent in sentences:
        content_only = re.sub(r"[\W_]+", "", sent, flags=re.UNICODE)
        if not content_only:
            continue

        words = sent.split()
        current_words = []

        for w in words:
            test_block = " ".join(current_words + [w])

            if not is_valid_block_text(test_block, max_chars_block, max_line_len, max_lines):
                if current_words:
                    blocks.append(" ".join(current_words))
                    current_words = [w]
                else:
                    current_words = [w]
            else:
                current_words.append(w)

        if current_words:
            blocks.append(" ".join(current_words))

    return blocks


def reading_speed(block):
    duration = block["end"] - block["start"]
    if duration <= 0:
        return float("inf")
    return len(block["text"]) / duration


def refresh_block(block):
    block = block.copy()
    block["text"] = clean_text_for_subs(block["text"])
    block["lines"] = split_text_into_lines(block["text"])
    return block


def block_is_valid(block):
    return is_valid_block_text(block["text"])


def adjust_timings_for_reading_speed(
    blocks,
    max_reading_speed: float = MAX_READING_SPEED,
    min_gap: float = MIN_GAP_BETWEEN_SUBS,
):
    if not blocks:
        return blocks

    adjusted = [b.copy() for b in blocks]

    for i, block in enumerate(adjusted):
        text_len = len(block["text"])
        current_start = block["start"]
        current_end = block["end"]
        current_duration = current_end - current_start

        if current_duration <= 0:
            continue

        current_speed = text_len / current_duration

        if current_speed <= max_reading_speed:
            continue

        needed_duration = text_len / max_reading_speed
        desired_end = current_start + needed_duration

        if i < len(adjusted) - 1:
            next_start = adjusted[i + 1]["start"]
            max_possible_end = next_start - min_gap
        else:
            max_possible_end = desired_end

        if max_possible_end > current_end:
            new_end = min(desired_end, max_possible_end)

            if new_end > current_end:
                block["end"] = new_end
                block["timing_adjusted"] = True
            else:
                block["timing_adjusted"] = False
        else:
            block["timing_adjusted"] = False

    return adjusted


def segments_to_blocks(
    segments,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
    min_block_dur: float = MIN_BLOCK_DUR,
    adjust_reading_time: bool = True,
):
    subtitle_blocks = []

    for seg in segments:
        seg_start = seg["start"]
        seg_end = seg["end"]
        seg_text = seg["text"].strip()

        if not seg_text:
            continue

        blocks = split_segment_text_into_blocks(
            seg_text,
            max_chars_block=max_chars_block,
            max_line_len=max_line_len,
            max_lines=max_lines,
        )

        if not blocks:
            continue

        seg_duration = seg_end - seg_start
        total_chars = sum(len(b) for b in blocks)

        if total_chars == 0 or seg_duration <= 0:
            continue

        n_blocks = len(blocks)
        base_durs = [(len(b) / total_chars) * seg_duration for b in blocks]

        if seg_duration >= n_blocks * min_block_dur:
            extra_time = seg_duration - n_blocks * min_block_dur
            base_sum = sum(base_durs) or 1.0
            block_durs = [
                min_block_dur + extra_time * (bd / base_sum)
                for bd in base_durs
            ]
        else:
            block_durs = base_durs

        dur_sum = sum(block_durs)

        if dur_sum > 0:
            scale = seg_duration / dur_sum
            block_durs = [d * scale for d in block_durs]

        current_time = seg_start

        for block_text, block_dur in zip(blocks, block_durs):
            block_end = current_time + block_dur

            subtitle_blocks.append({
                "start": current_time,
                "end": block_end,
                "original_start": current_time,
                "original_end": block_end,
                "text": block_text,
                "lines": split_text_into_lines(block_text),
                "timing_adjusted": False,
            })

            current_time = block_end

    if adjust_reading_time:
        subtitle_blocks = adjust_timings_for_reading_speed(subtitle_blocks)

    return subtitle_blocks

Синтаксическая сегментация на субтитры

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")
def is_bad_boundary(left_text, right_text):
    left_doc = nlp(left_text)
    right_doc = nlp(right_text)

    if len(left_doc) == 0 or len(right_doc) == 0:
        return False

    left_last = left_doc[-1]
    right_first = right_doc[0]

    left_word = left_last.text.lower()
    right_pos = right_first.pos_

    if left_last.pos_ == "ADP" and right_pos in {"NOUN", "PROPN", "PRON", "DET", "ADJ"}:
        return True
    if left_last.pos_ == "DET" and right_pos in {"NOUN", "PROPN", "ADJ"}:
        return True
    if left_word == "to" and right_pos == "VERB":
        return True
    if left_word in {"not", "n't"} and right_pos in {"VERB", "AUX", "ADJ"}:
        return True
    if left_last.pos_ == "AUX" and right_pos in {"VERB", "AUX"}:
        return True
    if left_last.pos_ == "ADJ" and right_pos in {"NOUN", "PROPN"}:
        return True
    if left_last.pos_ == "PROPN" and right_pos == "PROPN":
        return True
    if left_last.pos_ == "PRON" and right_pos in {"VERB", "AUX"}:
        return True

    return False


def syntax_boundary_refinement(blocks):
    if len(blocks) <= 1:
        return blocks

    refined = []
    i = 0

    while i < len(blocks):
        current = refresh_block(blocks[i])

        if i < len(blocks) - 1:
            next_block = refresh_block(blocks[i + 1])

            if is_bad_boundary(current["text"], next_block["text"]):
                candidate = {
                    "start": current["start"],
                    "end": next_block["end"],
                    "original_start": current.get("original_start", current["start"]),
                    "original_end": next_block.get("original_end", next_block["end"]),
                    "text": current["text"] + " " + next_block["text"],
                    "timing_adjusted": (
                        current.get("timing_adjusted", False)
                        or next_block.get("timing_adjusted", False)
                    ),
                }

                candidate = refresh_block(candidate)

                if block_is_valid(candidate) and reading_speed(candidate) <= MAX_READING_SPEED:
                    refined.append(candidate)
                    i += 2
                    continue

        refined.append(current)
        i += 1

    return refined


def build_improved_en_blocks(segments):
    blocks = segments_to_blocks(
        segments,
        max_chars_block=MAX_CHARS_BLOCK,
        max_line_len=MAX_LINE_LEN,
        max_lines=MAX_LINES,
        min_block_dur=MIN_BLOCK_DUR,
        adjust_reading_time=True,
    )

    blocks = syntax_boundary_refinement(blocks)

    blocks = adjust_timings_for_reading_speed(
        blocks,
        max_reading_speed=MAX_READING_SPEED,
        min_gap=MIN_GAP_BETWEEN_SUBS,
    )

    return blocks

Подготовка субтитров

In [8]:
prepared_items = []

for item in whisper_results:
    folder = item["folder"]
    segments = item["segments"]

    start = time.time()

    en_blocks = build_improved_en_blocks(segments)

    segmentation_time = time.time() - start

    prepared_items.append({
        "folder": folder,
        "en_blocks": en_blocks,
        "en_texts": [b["text"] for b in en_blocks],
        "segmentation_time_sec": round(segmentation_time, 4),
        "num_subtitles": len(en_blocks),
    })

    print(
        folder,
        "субтитров:", len(en_blocks),
        "сегментация:", round(segmentation_time, 4), "сек"
    )

print("Готово. Видео подготовлено:", len(prepared_items))

video1 субтитров: 264 сегментация: 4.6386 сек
video10 субтитров: 29 сегментация: 0.3123 сек
video11 субтитров: 77 сегментация: 0.9327 сек
video12 субтитров: 10 сегментация: 0.089 сек
video13 субтитров: 22 сегментация: 0.2326 сек
video14 субтитров: 15 сегментация: 0.1772 сек
video15 субтитров: 18 сегментация: 0.1791 сек
video2 субтитров: 123 сегментация: 1.3604 сек
video3 субтитров: 376 сегментация: 5.2396 сек
video4 субтитров: 249 сегментация: 3.2256 сек
video5 субтитров: 376 сегментация: 5.308 сек
video6 субтитров: 33 сегментация: 0.3711 сек
video7 субтитров: 408 сегментация: 8.6574 сек
video8 субтитров: 78 сегментация: 1.4601 сек
video9 субтитров: 831 сегментация: 13.3756 сек
Готово. Видео подготовлено: 15


Подготовка эталонного перевода

In [10]:
def strip_srt(text: str) -> str:
    lines = text.splitlines()
    clean_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue
        if line.isdigit():
            continue
        if "-->" in line:
            continue

        clean_lines.append(line)

    return " ".join(clean_lines)


def find_reference_ru(folder_name):
    folder_path = dataset_path / folder_name

    candidates = []

    for pattern in [
        "*reference*ru*.txt",
        "*ref*ru*.txt",
        "*ru*.txt",
        "*reference*.srt",
        "*ref*.srt",
        "*ru*.srt",
    ]:
        candidates.extend(folder_path.glob(pattern))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        return None, None

    ref_path = candidates[0]
    text = ref_path.read_text(encoding="utf-8")

    if ref_path.suffix.lower() == ".srt":
        text = strip_srt(text)

    return ref_path, text.strip()


references = {}

for item in prepared_items:
    ref_path, ref_text = find_reference_ru(item["folder"])

    if ref_text:
        references[item["folder"]] = ref_text
        print(item["folder"], "->", ref_path.name)
    else:
        print("Не найден эталонный перевод:", item["folder"])

print("Эталонных переводов найдено:", len(references))

video1 -> ru.srt
video10 -> ru.srt
video11 -> ru.srt
video12 -> ru.srt
video13 -> ru.srt
video14 -> ru.srt
video15 -> ru.srt
video2 -> ru.srt
video3 -> ru.srt
video4 -> ru.srt
video5 -> ru.srt
video6 -> ru.srt
video7 -> ru.srt
video8 -> ru.srt
video9 -> ru.srt
Эталонных переводов найдено: 15


Общие функции перевода

In [14]:
import torch
import gc
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cpu"


def get_model_size_info(model):
    params = sum(p.numel() for p in model.parameters())
    size_mb_fp32 = params * 4 / 1024 / 1024

    return {
        "params_mln": round(params / 1_000_000, 1),
        "approx_size_mb_fp32": round(size_mb_fp32, 1),
    }


def translate_texts(
    texts,
    tokenizer,
    model,
    model_type,
    batch_size=4,
    max_new_tokens=128,
    src_lang=None,
    tgt_lang=None,
    verbose=False,
):
    translated = []

    if model_type == "nllb":
        tokenizer.src_lang = src_lang
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    else:
        forced_bos_token_id = None

    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        if verbose:
            print(f"Батч {i // batch_size + 1} / {total_batches}")

        batch_texts = texts[i:i + batch_size]
        batch_texts = [t if t.strip() else "." for t in batch_texts]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(DEVICE)

        with torch.no_grad():
            if model_type == "nllb":
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=5,
                    forced_bos_token_id=forced_bos_token_id,
                )
            else:
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=5,
                    no_repeat_ngram_size=3,
                )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        translated.extend([d.strip() for d in decoded])

    return translated

Фунции оценки качества

In [17]:
import sacrebleu
all_results = []
translations_by_model = {}


def run_translation_experiment_for_model(
    model_label,
    model_name,
    model_type,
    batch_size=4,
    src_lang=None,
    tgt_lang=None,
):
    print("=" * 50)
    print("Загрузка модели:", model_label)
    print("=" * 50)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    size_info = get_model_size_info(model)

    model_translations = {}

    for item in prepared_items:
        folder = item["folder"]
        en_texts = item["en_texts"]

        print("\nВидео:", folder)
        print("Количество субтитров:", len(en_texts))

        start = time.time()

        ru_texts = translate_texts(
            en_texts,
            tokenizer,
            model,
            model_type=model_type,
            batch_size=batch_size,
            max_new_tokens=128,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            verbose=False,
        )

        translation_time = time.time() - start

        hypothesis = " ".join(ru_texts)
        reference = references.get(folder)

        if reference:
            bleu = sacrebleu.corpus_bleu([hypothesis], [[reference]]).score
            chrf = sacrebleu.corpus_chrf([hypothesis], [[reference]]).score
        else:
            bleu = None
            chrf = None

        model_translations[folder] = ru_texts

        all_results.append({
            "folder": folder,
            "model": model_label,
            "BLEU": round(bleu, 4) if bleu is not None else None,
            "chrF": round(chrf, 4) if chrf is not None else None,
            "translation_time_sec": round(translation_time, 4),
            "segmentation_time_sec": item["segmentation_time_sec"],
            "total_time_sec": round(translation_time + item["segmentation_time_sec"], 4),
            "num_subtitles": item["num_subtitles"],
            "avg_time_per_subtitle_sec": round(
                translation_time / max(1, item["num_subtitles"]),
                4
            ),
            "params_mln": size_info["params_mln"],
            "approx_size_mb_fp32": size_info["approx_size_mb_fp32"],
        })

        print("BLEU:", bleu)
        print("chrF:", chrf)
        print("Время перевода:", round(translation_time, 4), "сек")

    translations_by_model[model_label] = model_translations

    del tokenizer
    del model
    gc.collect()

    print("\nМодель выгружена из памяти:", model_label)

Запуск Opus-MT

In [18]:
run_translation_experiment_for_model(
    model_label="Opus-MT",
    model_name="Helsinki-NLP/opus-mt-en-ru",
    model_type="opus",
    batch_size=4,
)

Загрузка модели: Opus-MT


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Видео: video1
Количество субтитров: 264
BLEU: 17.838024204195023
chrF: 60.70026889971193
Время перевода: 139.0026 сек

Видео: video10
Количество субтитров: 29
BLEU: 19.9440400646991
chrF: 51.39556374360962
Время перевода: 13.0807 сек

Видео: video11
Количество субтитров: 77
BLEU: 18.64387881385174
chrF: 55.869441490388795
Время перевода: 54.6148 сек

Видео: video12
Количество субтитров: 10
BLEU: 31.556981131410236
chrF: 57.728650590499385
Время перевода: 3.7593 сек

Видео: video13
Количество субтитров: 22
BLEU: 14.380330735679555
chrF: 45.94791713792515
Время перевода: 12.786 сек

Видео: video14
Количество субтитров: 15
BLEU: 36.4383108825434
chrF: 62.77717426746317
Время перевода: 6.4161 сек

Видео: video15
Количество субтитров: 18
BLEU: 21.689273274569707
chrF: 56.085018424772684
Время перевода: 8.4243 сек

Видео: video2
Количество субтитров: 123
BLEU: 20.963423488196756
chrF: 52.394633677418
Время перевода: 62.2903 сек

Видео: video3
Количество субтитров: 376
BLEU: 17.9391001079162

In [19]:
import pandas as pd

results_df = pd.DataFrame(all_results)
results_df

,folder,model,BLEU,chrF,translation_time_sec,segmentation_time_sec,total_time_sec,num_subtitles,avg_time_per_subtitle_sec,params_mln,approx_size_mb_fp32
0,video1,Opus-MT,17.8380,60.7003,139.0026,4.6386,143.6412,264,0.5265,140.7,536.7
1,video10,Opus-MT,19.9440,51.3956,13.0807,0.3123,13.3930,29,0.4511,140.7,536.7
2,video11,Opus-MT,18.6439,55.8694,54.6148,0.9327,55.5475,77,0.7093,140.7,536.7
3,video12,Opus-MT,31.5570,57.7287,3.7593,0.0890,3.8483,10,0.3759,140.7,536.7
4,video13,Opus-MT,14.3803,45.9479,12.7860,0.2326,13.0186,22,0.5812,140.7,536.7
5,video14,Opus-MT,36.4383,62.7772,6.4161,0.1772,6.5933,15,0.4277,140.7,536.7
6,video15,Opus-MT,21.6893,56.0850,8.4243,0.1791,8.6034,18,0.4680,140.7,536.7
7,video2,Opus-MT,20.9634,52.3946,62.2903,1.3604,63.6507,123,0.5064,140.7,536.7
8,video3,Opus-MT,17.9391,58.8501,183.0707,5.2396,188.3103,376,0.4869,140.7,536.7
9,video4,Opus-MT,24.5537,68.0720,125.7108,3.2256,128.9364,249,0.5049,140.7,536.7


Запуск NLLB-600M

In [20]:
run_translation_experiment_for_model(
    model_label="NLLB-600M",
    model_name="facebook/nllb-200-distilled-600M",
    model_type="nllb",
    batch_size=2,
    src_lang="eng_Latn",
    tgt_lang="rus_Cyrl",
)

Загрузка модели: NLLB-600M


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


Видео: video1
Количество субтитров: 264
BLEU: 17.27192719268831
chrF: 61.81102827282607
Время перевода: 1474.7464 сек

Видео: video10
Количество субтитров: 29
BLEU: 17.15808751782828
chrF: 48.209027438263476
Время перевода: 111.9195 сек

Видео: video11
Количество субтитров: 77
BLEU: 17.803118185809517
chrF: 58.23787258184545
Время перевода: 581.3908 сек

Видео: video12
Количество субтитров: 10
BLEU: 23.793665482062618
chrF: 57.36409886168373
Время перевода: 40.136 сек

Видео: video13
Количество субтитров: 22
BLEU: 3.5641714976999017
chrF: 39.90325278134078
Время перевода: 219.0488 сек

Видео: video14
Количество субтитров: 15
BLEU: 25.847055226898416
chrF: 60.17042364137303
Время перевода: 69.555 сек

Видео: video15
Количество субтитров: 18
BLEU: 26.955322115635223
chrF: 60.61423325934633
Время перевода: 64.706 сек

Видео: video2
Количество субтитров: 123
BLEU: 18.770746826543125
chrF: 52.889991328272735
Время перевода: 641.0391 сек

Видео: video3
Количество субтитров: 376
BLEU: 18.079

Результаты

In [21]:
results_df = pd.DataFrame(all_results)
results_df

,folder,model,BLEU,chrF,translation_time_sec,segmentation_time_sec,total_time_sec,num_subtitles,avg_time_per_subtitle_sec,params_mln,approx_size_mb_fp32
0,video1,Opus-MT,17.8380,60.7003,139.0026,4.6386,143.6412,264,0.5265,140.7,536.7
1,video10,Opus-MT,19.9440,51.3956,13.0807,0.3123,13.3930,29,0.4511,140.7,536.7
2,video11,Opus-MT,18.6439,55.8694,54.6148,0.9327,55.5475,77,0.7093,140.7,536.7
3,video12,Opus-MT,31.5570,57.7287,3.7593,0.0890,3.8483,10,0.3759,140.7,536.7
4,video13,Opus-MT,14.3803,45.9479,12.7860,0.2326,13.0186,22,0.5812,140.7,536.7
5,video14,Opus-MT,36.4383,62.7772,6.4161,0.1772,6.5933,15,0.4277,140.7,536.7
6,video15,Opus-MT,21.6893,56.0850,8.4243,0.1791,8.6034,18,0.4680,140.7,536.7
7,video2,Opus-MT,20.9634,52.3946,62.2903,1.3604,63.6507,123,0.5064,140.7,536.7
8,video3,Opus-MT,17.9391,58.8501,183.0707,5.2396,188.3103,376,0.4869,140.7,536.7
9,video4,Opus-MT,24.5537,68.0720,125.7108,3.2256,128.9364,249,0.5049,140.7,536.7


In [22]:
summary_df = (
    results_df
    .groupby("model", as_index=False)
    .agg({
        "BLEU": "mean",
        "chrF": "mean",
        "translation_time_sec": "mean",
        "total_time_sec": "mean",
        "avg_time_per_subtitle_sec": "mean",
        "params_mln": "first",
        "approx_size_mb_fp32": "first",
    })
)

summary_df = summary_df.sort_values(
    by=["chrF", "BLEU"],
    ascending=False,
)

summary_df

,model,BLEU,chrF,translation_time_sec,total_time_sec,avg_time_per_subtitle_sec,params_mln,approx_size_mb_fp32
1,Opus-MT,20.770693,57.275353,96.407800,99.445087,0.506713,140.7,536.7
0,NLLB-600M,18.844300,57.207753,1009.099393,1012.136680,5.418780,1402.1,5348.7
